# Designing the Tingen City Background — End-to-End Walkthrough

A rough but complete tour of how the explorable **city background** is built, from a blank prompt
to a composed Godot scene where the buildings are solid (colliders) and the player / agents route
around them. Written for Artemis and Maxine ahead of our navigation chat, and for the rest of the lab.

The pipeline is four moves:

1. **Prompt** a true top-down city (style + important features).
2. **Relabel** the place names to *Lord of Mysteries* canon.
3. **Split into two layers** from that one image:
   - **strip the buildings → background only** (bare streets/ground), and
   - **ask for the buildings only** (isolated, transparent).
4. **Compose in the Godot scene** — background + building sprites, each building wrapped in a
   collider so it's solid.

Deep reference: [`MAP_TUTORIAL.md`](MAP_TUTORIAL.md); art rules: [`STYLE_GUIDE.md`](STYLE_GUIDE.md).

## Tooling & attribution (please read)

Two image tools were used, and it matters which produced what:

- **Scripted API pipeline** — `generate_tingen_image2.py`, calling OpenAI's **`gpt-image-1`** Images
  API. Reproducible path; every call is logged to `manifest_image2.json`.
- **ChatGPT in-app image generator (GPT-Image, the "image 2" tool in the ChatGPT UI)** — used
  *interactively* by Mark for a few edits that were faster to iterate by hand than to script.
  **Not captured in any script or manifest**, so it's credited explicitly here:
  - `my_assets/map_bg.png` (the **background-only / strip** in Step 3) was produced in the ChatGPT
    in-app image tool, not the API.
  - Several one-off `my_assets/` exports (e.g. `ChatGPT Image Jun 18, 2026, 02_57_26 PM.png`) came
    from the same in-app tool.

> Same underlying model family as the API; the difference is *interactive chat* vs. *scripted call*.
> Where a step was done by hand in ChatGPT, the notebook says so inline.

## The one constraint that shapes everything: `gpt-image-1` has no seed

You cannot reproduce an image with a fixed random seed — there isn't one. The **only** levers for
consistency are (1) the **reference image(s)** you pass and (2) **`input_fidelity`** (`low` | `high`),
how tightly the output clings to the reference.

Two endpoints, chosen by whether you have a reference:

| Endpoint | When | Body |
|---|---|---|
| `POST /v1/images/edits` | you pass a reference image | **multipart**, `image[]=@ref.png` |
| `POST /v1/images/generations` | no reference (text only) | **JSON** |

Cost (approx): `low` ~$0.02 · `medium` ~$0.05 · `high` ~$0.25 per image. **Draft on `low`, finalize
on `high`.** Every step below is an *edit* (we always have a reference once the first map exists).

## 0. Setup

In [ ]:
# pip install requests pillow numpy   (scipy optional, speeds up the keying step)
import os, base64, requests
from pathlib import Path
from IPython.display import Image as ShowImage, display

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')  # or load from the .env the pipeline reads
# Run this notebook from asset-gen/ so the relative image paths resolve.

def gen_image(prompt, ref=None, size='1536x1024', fidelity='high',
              quality='high', background='opaque', out='out.png'):
    """Minimal version of what generate_tingen_image2.py does under the hood.
    With a ref -> /images/edits (multipart). Without -> /images/generations (JSON).
    Use background='transparent' for the buildings-only layer in Step 3b."""
    headers = {'Authorization': f'Bearer {OPENAI_API_KEY}'}
    if ref:
        files = {'image[]': (Path(ref).name, open(ref, 'rb'), 'image/png')}
        data = {'model': 'gpt-image-1', 'prompt': prompt, 'size': size,
                'quality': quality, 'background': background,
                'output_format': 'png', 'input_fidelity': fidelity, 'n': 1}
        r = requests.post('https://api.openai.com/v1/images/edits',
                          headers=headers, files=files, data=data, timeout=1800)
    else:
        body = {'model': 'gpt-image-1', 'prompt': prompt, 'size': size,
                'quality': quality, 'background': background,
                'output_format': 'png', 'n': 1}
        r = requests.post('https://api.openai.com/v1/images/generations',
                          headers={**headers, 'Content-Type': 'application/json'},
                          json=body, timeout=1800)
    r.raise_for_status()
    Path(out).write_bytes(base64.b64decode(r.json()['data'][0]['b64_json']))
    return out

# NOTE: latency is wildly variable (60s typical; one background once took ~24 min).
# In production, run generate_tingen_image2.py as a background job, not a foreground call.

## 1. Prompt the top-down city (style + important features)

**The hard problem:** a prompt asking for a "top-down city" *always* drifts to a ¾/oblique angle.
We tried 7 style variants; prose alone never holds a true 90° overhead.

**The fix:** feed a **flat-overhead reference** and set **`input_fidelity=high`** — high fidelity
locks the straight-down angle to the reference. Then load the prompt with **style + the important
features** the city must contain.

- **Angle:** `TRUE flat 90-degree birds-eye … NO perspective, NO tilt, NO isometric, NO building fronts`.
- **Style (from `STYLE_GUIDE.md`):** late-Victorian gaslamp fantasy, muted fog-gray base + warm
  gaslight; **pale cool-grey streets lighter than every roof**, roofs a **patchwork** of terracotta /
  charcoal-slate / verdigris so adjacent buildings differ in hue *and* brightness (this is the contrast
  fix — the first pass came out a monochrome orange wash). Negatives: `NOT a monochrome wash, NOT one
  hue, NOT low-contrast`.
- **Important features:** the river splitting districts, Saint Selena's twin-spired cathedral, the
  university quad, Iron Cross market, the docks — placed where canon wants them.

In [ ]:
display(ShowImage(filename='ref/tingen_map.png', width=460))   # flat ref that locks the 90 degree angle

In [ ]:
# Via the pipeline (preferred -- resumable, logged):
#   python3 generate_tingen_image2.py --category backgrounds --treatment district_flat --quality high
#
# Equivalent direct call (this is BG_DISTRICT_FLAT in the script):
CITY_PROMPT = (
    'a large highly detailed city district map seen from directly straight above at a TRUE flat '
    '90-degree birds-eye angle, only rooftops and streets seen flat from above, NO perspective, '
    'NO tilt, NO isometric angle, NO building fronts. Late-Victorian gaslamp fantasy, muted fog-gray '
    'palette with warm gaslight; PALE cool-grey streets, lighter than every roof; rooftops a vivid '
    'patchwork of terracotta, charcoal-slate and verdigris so adjacent buildings differ in hue and '
    'brightness. A river splitting the city into districts, a twin-spired Gothic cathedral, a red-brick '
    'university quad, a market street, docks. NOT a monochrome wash, NOT one hue, NOT low-contrast.'
)
# gen_image(CITY_PROMPT, ref='ref/tingen_map.png', size='1536x1024', fidelity='high', out='city_v1.png')

## 2. Relabel place names to canon

The generated city has fictional/garbled labels. Relabel to canon with an **image-to-image edit**:
pass the city image *itself* as the reference at `input_fidelity=high` so art + layout are preserved,
and change **only** the text.

⚠ **The text caveat:** AI image tools garble lettering (our first map literally produced "GREENMARKET
SQUARK"). Prefer to **generate label-free and add the ~5 district names by hand** (Preview / Photopea),
or label only a handful via the API and fix spelling manually. Verify against `../tingen_npc_roster.md`
and the GDD §6.2 before baking them in.

In [ ]:
RELABEL = (
    'Keep this exact city - same style, same layout, same top-down angle. '
    'Change ONLY the text labels to: Iron Cross Market, The Laughing Eel, '
    "Saint Selena's Cathedral, the Docks. Spell every label exactly as written. "
    'Make no other changes.'
)
# gen_image(RELABEL, ref='city_v1.png', fidelity='high', out='my_assets/map_v3.png')

display(ShowImage(filename='my_assets/map_v3.png', width=520))   # labeled city

## 3. Split the one image into two layers

This is the core trick: from the single labeled city image, produce **two complementary layers** with
two edits. Both pass the labeled city as the reference at `input_fidelity=high` so the layout/geometry
stays pixel-aligned between the layers — that alignment is what lets them recompose cleanly.

### 3a. Strip the buildings → background only
Ask it to remove every building and keep only the walkable ground. The empty lots barely need detail —
they'll sit *under* the buildings — they just need to read as streets.

Two routes:
- **A. API edit (scriptable):** the `STRIP_BG` prompt below.
- **B. ChatGPT in-app image tool (what actually made the current `map_bg.png`):** upload the city image,
  ask to remove the buildings, done by hand. *This is the in-app GPT-Image path credited at the top.*

In [ ]:
STRIP_BG = (
    'Remove ALL buildings, rooftops and labels. Keep ONLY the cobblestone streets, lanes, squares '
    'and the empty ground between them, in the exact same layout. Flat top-down. '
    'Empty paved/dirt lots where buildings were.'
)
# gen_image(STRIP_BG, ref='my_assets/map_v3.png', size='1254x1254', fidelity='high',
#           background='opaque', out='my_assets/map_bg.png')

display(ShowImage(filename='my_assets/map_bg.png', width=520))   # background-only (made via ChatGPT in-app tool)

### 3b. Ask for the buildings only → transparent building layer
The mirror edit: keep **only** the buildings, in their exact positions, on a **transparent** background
(`background='transparent'`), no streets/ground/labels. Because it shares the reference + layout with 3a,
the buildings drop straight back onto the background at the same coordinates.

Buildings sometimes still arrive on a near-white halo; `key_building.py` cleans the alpha in three steps —
**KEY** (border-connected near-white → transparent, so bright pixels *inside* a building survive), **ERODE**
(shrink the edge to kill the fringe ring), **BLEED** (fill transparent RGB with the nearest opaque colour so
the engine's linear filtering never samples white). Each building is a self-contained compound (building +
its immediate grounds), e.g. `St_Selena_Chapel_v2.png` = church + churchyard + fence.

In [ ]:
BUILDINGS_ONLY = (
    'Keep ONLY the buildings and rooftops, in their exact same positions and footprints. '
    'Remove ALL streets, ground and labels. Transparent background. Flat top-down, same layout.'
)
# gen_image(BUILDINGS_ONLY, ref='my_assets/map_v3.png', size='1254x1254', fidelity='high',
#           background='transparent', out='buildings_layer.png')
#
# Clean the alpha if a building came out on white instead of transparent:
#   python3 key_building.py my_assets/St_Selena_Chapel_v2.png tingen/assets/props/chapel.png --white 222 --erode 2

display(ShowImage(filename='my_assets/St_Selena_Chapel_v2.png', width=300))   # example building compound

## 4. Compose in the Godot scene — background + buildings + colliders

Now assemble the two layers in the `.tscn` and make the buildings **solid** so the player and agents can't
walk through them and route around instead. The real composed scene is `tingen/scenes/CityBlocks.tscn`:
a single walkable `Ground` polygon (the background layer) plus a `Buildings` node holding **53** building
bodies, each a `StaticBody2D` with a `Polygon2D` (the visual) **and** a `CollisionPolygon2D` (the footprint):

```gdscript
[node name="Ground" type="Polygon2D" parent="."]
polygon = PackedVector2Array(0, 0, 2400, 0, 2400, 1700, 0, 1700)        # the background / walkable floor

[node name="B_r0_c0" type="StaticBody2D" parent="Buildings"]            # one building
[node name="Polygon2D" type="Polygon2D" parent="Buildings/B_r0_c0"]     # building sprite/visual
polygon = PackedVector2Array(-76.8, -58.8, 76.8, -58.8, 76.8, 58.8, -76.8, 58.8)
[node name="CollisionPolygon2D" type="CollisionPolygon2D" parent="Buildings/B_r0_c0"]  # THE collider
polygon = PackedVector2Array(-76.8, -58.8, 76.8, -58.8, 76.8, 58.8, -76.8, 58.8)
```

`City.tscn` adds a `Solids` body for the map edges + the cathedral, and `Area2D` door triggers
(`ChapelDoor`, `NeilHomeDoor`) for entering interiors.

**Recipe:**
- **Background:** `map_bg.png` as the ground `Sprite2D`/`Polygon2D` (`texture_filter = Linear` for smooth
  upscale of painted art).
- **Buildings:** the keyed building sprite(s) stamped at their footprint position; each wrapped in a
  `StaticBody2D` + `CollisionPolygon2D` so it's a solid obstacle.
- Keep every building at the **same iso angle** and a fixed Klein-to-doorway scale, and hold the camera
  at **gameplay zoom** (the iso-building-on-top-down-ground mismatch only shows when zoomed out).

The collider footprints are what the player and NPC agents collide with, so navigation routes *around*
the buildings — the streets between them are the walkable space.

## Why this is the bridge to Maxine and Artemis

Stripped to its bones, the composed scene is exactly a **navigation problem**:
- a **walkable field** — the background layer (`map_bg`), and
- a set of **obstacle footprints** — the per-building `CollisionPolygon2D`s.

- **Maxine (maze navigation):** the building-footprint layout *is* a maze — same routing / obstacle-
  avoidance, just sourced from a generated city instead of a synthetic maze.
- **Artemis (VLM navigation / AI2 synthetic maps):** the `map_v3` (labeled, semantic) + `map_bg` (bare
  walkable) + buildings-only layers are a natural VLM-navigation input, and the prompt→relabel→split→
  compose recipe is a way to mint synthetic-city + obstacle-layout pairs.

## Run order & file map

| Stage | Command / file | Output |
|---|---|---|
| 1 prompt | `generate_tingen_image2.py --category backgrounds --treatment district_flat` | generated city |
| 2 relabel | image-to-image edit (prefer manual text) | `my_assets/map_v3.png` |
| 3a strip → background | API edit **or ChatGPT in-app tool** | `my_assets/map_bg.png` |
| 3b buildings only | API edit (transparent) → `key_building.py` | building layer / sprites |
| 4 compose + colliders | `tingen/scenes/CityBlocks.tscn` | `Ground` + `Buildings/*` StaticBody2D |

### TL;DR
flat ref + `input_fidelity=high` to lock the 90° angle → prompt the city with style + important features
→ relabel to canon (prefer manual text) → split the one image into a **background-only** layer and a
**buildings-only** layer (two aligned edits) → compose in the Godot scene: `map_bg` as the walkable ground,
building sprites stamped on top, each in a `StaticBody2D` + `CollisionPolygon2D` so it's solid and agents
route around it.

*Image credits: `gpt-image-1` API via `generate_tingen_image2.py`, except `map_bg.png` and assorted
`my_assets/` exports, made with the ChatGPT in-app image generator (GPT-Image / "image 2").*